# Amazon Bedrock AgentCore Policy - Getting Started Demo

## Overview

Welcome to the Amazon Bedrock AgentCore Policy hands-on demo! This notebook will guide you through the complete workflow of setting up and testing policy-based security controls for AI agent interactions.

### What is AgentCore Policy?

Amazon Bedrock AgentCore Policy enables developers to define and enforce security controls for AI agent interactions with tools by creating a protective boundary ("safety box") around agent operations. AI agents can dynamically adapt to solve complex problems, but this flexibility introduces security challenges:

- **Data Leakage**: Agents may inadvertently expose private information
- **Business Rule Violations**: Agents might misinterpret or bypass business rules
- **Authority Overreach**: Agents could act outside their intended scope

Policy intercepts inbound agent traffic through AgentCore Gateways and evaluates each request against defined policies before allowing tool access.

### Key Benefits

✅ **Declarative Security**: Define policies using Cedar language, not code  
✅ **Runtime Enforcement**: Policies are evaluated in real-time  
✅ **Fine-Grained Control**: From coarse restrictions to detailed transaction limits  
✅ **Separation of Concerns**: Security logic lives outside agent code  
✅ **Enterprise Scale**: Deploy autonomous agents safely in production  

---

## Demo Architecture

```
┌─────────────┐
│   AI Agent  │
└──────┬──────┘
       │
       │ Tool Call Request
       ▼
┌─────────────────────┐
│  AgentCore Gateway  │
│  + OAuth Auth       │
└──────┬──────────────┘
       │
       │ Policy Check
       ▼
┌─────────────────────┐
│   Policy Engine     │
│   (Cedar Policies)  │
└──────┬──────────────┘
       │
       │ ALLOW / DENY
       ▼
┌─────────────────────┐
│   Lambda Target     │
│   (RefundTool)      │
└─────────────────────┘
```

---

## What You'll Learn

In this demo, you will:

1. **Setup Infrastructure**: Create a Gateway with Lambda targets
2. **Create Policy Engine**: Initialize a policy engine for your gateway
3. **Define Policies**: Write Cedar policies to control access
4. **Test Enforcement**: Verify policies work with real agent requests
5. **Understand Results**: Interpret ALLOW and DENY scenarios

---

## Prerequisites

Before starting, ensure you have:

- ✅ AWS CLI configured with appropriate credentials
- ✅ Python 3.10+
---

## Demo Scenario: Refund Processing

We'll implement a **refund processing system** with policy controls:

- **Tool**: `RefundTool` - Processes customer refunds
- **Parameters**: `amount` (integer), `orderId` (string)
- **Policy Rule**: Only allow refunds under $1000
- **Test Cases**: 
  - ✅ $200 refund (should be ALLOWED)
  - ❌ $2000 refund (should be DENIED)

Let's get started! 🚀

---

# Step 0: Environment Setup

First, let's install dependencies and verify our environment.

In [ ]:
# Install required packages
!pip3 install -r requirements.txt --quiet

In [ ]:
# Import required libraries
import json
import time
import boto3
from pathlib import Path

# Verify region
session = boto3.Session()
region = session.region_name or 'us-east-1'

# Verify AWS credentials
try:
    sts = session.client('sts')
    identity = sts.get_caller_identity()
    print("✅ AWS Credentials Verified")
    print(f"   Account: {identity['Account']}")
    print(f"   User/Role: {identity['Arn']}")
except Exception as e:
    print(f"❌ AWS Credentials Error: {e}")
    print("   Please configure AWS CLI with: aws configure")

---

# Step 1: Create Gateway and Lambda Function

We'll create the gateway infrastructure using the starter toolkit utilities.

## Why Do We Need a Lambda Function?

The Lambda function serves as the **backend tool** that our AI agent will call through the gateway. In this demo:

- **Lambda = RefundTool**: Processes customer refund requests
- **Bedrock AgentCore Gateway** = provides customers a way to turn their existing AWS Lambda functions into fully-managed MCP servers without needing to manage infra or hosting. Gateway will provide a uniform Model Context Protocol (MCP) interface across all these tools.
- **Policy Engine** = A policy engine is a collection of Cedar policies that evaluates and authorizes agent tool calls. The policy engine intercepts all requests at the gateway boundary and determines whether to allow or deny each action based on the defined policies. This provides deterministic authorization outside of the agent's code, ensuring consistent security enforcement regardless of how the agent is implemented.

Without the Lambda, there would be no actual tool for the agent to invoke. The policy engine (Step 2) will protect this Lambda by evaluating requests before they reach it.

## What Gets Created?

1. **Lambda Function**: Python function that processes refunds
2. **OAuth Authorizer**: Cognito-based authentication
3. **AgentCore Gateway**: MCP gateway that exposes Lambda as a tool

In [ ]:
# Import starter toolkit utilities
from bedrock_agentcore_starter_toolkit.operations.gateway.client import GatewayClient
from bedrock_agentcore_starter_toolkit.utils.lambda_utils import create_lambda_function

# Lambda function code for RefundTool (Python)
refund_lambda_code = """
def lambda_handler(event, context):
    amount = event.get('amount', 0)
    order_id = event.get('orderId', 'unknown')
    return {
        "status": "success",
        "message": f"Refund of ${amount} processed for order {order_id}",
        "amount": amount,
        "orderId": order_id
    }
"""

print("✅ Lambda code defined")
print("\n📋 Lambda Function Details:")
print("   Name: RefundTool")
print("   Runtime: Python 3.13")
print("   Handler: lambda_function.lambda_handler")
print("   Parameters: amount (int), orderId (string)")

# Initialize Gateway Client
print("\n🔧 Initializing Gateway Client...")
gateway_client = GatewayClient(region_name=region)

# Create OAuth authorizer with Cognito
print("\n🔐 Creating OAuth authorization server (Cognito)...")
cognito_response = gateway_client.create_oauth_authorizer_with_cognito("PolicyDemo")
print("✅ OAuth authorizer created")

# Create MCP Gateway
print("\n🚀 Creating AgentCore Gateway...")
gateway = gateway_client.create_mcp_gateway(
    name="RefundGateway",
    authorizer_config=cognito_response["authorizer_config"],
    enable_semantic_search=False
)

# Fix IAM permissions
gateway_client.fix_iam_permissions(gateway)
print("✅ Gateway created successfully")
print(f"   Gateway ID: {gateway['gatewayId']}")
print(f"   Gateway URL: {gateway['gatewayUrl']}")

# Wait for gateway to be ready
print("\n⏳ Waiting for gateway to be ready (30 seconds)...")
time.sleep(30)
print("✅ Gateway is ready")

# Create Lambda function using starter toolkit utility
print("\n🔧 Creating Lambda function...")
lambda_arn = create_lambda_function(
    session=session,
    logger=gateway_client.logger,
    function_name=f"RefundTool-{int(time.time())}",
    lambda_code=refund_lambda_code,
    runtime="python3.13",
    handler="lambda_function.lambda_handler",
    gateway_role_arn=gateway["roleArn"],
    description="Refund tool for policy demo"
)
print("✅ Lambda function created successfully")
print(f"   Lambda ARN: {lambda_arn}")

# Add Lambda target to gateway
print("\n🔗 Attaching Lambda to Gateway...")
gateway_client.create_mcp_gateway_target(
    gateway=gateway,
    name="RefundTarget",
    target_type="lambda",
    target_payload={
        "lambdaArn": lambda_arn,
        "toolSchema": {
            "inlinePayload": [
                {
                    "name": "refund",
                    "description": "Process a customer refund",
                    "inputSchema": {
                        "type": "object",
                        "properties": {
                            "amount": {"type": "integer", "description": "Refund amount in dollars"},
                            "orderId": {"type": "string", "description": "Order ID for the refund"}
                        },
                        "required": ["amount", "orderId"]
                    }
                }
            ]
        }
    }
)
print("✅ Lambda attached to gateway as RefundTarget")
print("   Tool Name: RefundTarget___refund")

# Store variables for later use
GATEWAY_ARN = gateway["gatewayArn"]
GATEWAY_ID = gateway["gatewayId"]
GATEWAY_URL = gateway["gatewayUrl"]

print("\n" + "="*60)
print("✅ Step 1 Complete: Gateway and Lambda Created")
print("="*60)
print(f"Gateway URL: {GATEWAY_URL}")
print(f"Lambda ARN: {lambda_arn}")
print("="*60)

---

# Step 2: Create Policy Engine

Now we'll create a Policy Engine that will hold our Cedar policies.

## What is a Policy Engine?

A Policy Engine evaluates requests against Cedar policies in real-time. It operates in two modes:
- **LOG_ONLY**: Evaluates but doesn't block (for testing)
- **ENFORCE**: Actively blocks non-compliant requests (for production)

The policy engine starts empty - we'll add policies using natural language in Step 3.

In [ ]:
# Import PolicyClient
from bedrock_agentcore_starter_toolkit.operations.policy.client import PolicyClient

# Create Policy Engine
print("🔧 Creating Policy Engine...")
policy_client = PolicyClient(region_name=region)
policy_engine = policy_client.create_or_get_policy_engine(
    name="RefundPolicyEngine",
    description="Policy engine for refund governance"
)
print("✅ Policy Engine created")
print(f"   Policy Engine ID: {policy_engine['policyEngineId']}")
print(f"   Policy Engine ARN: {policy_engine['policyEngineArn']}")

# Attach policy engine to gateway (in LOG_ONLY mode for now)
print("\n🔗 Attaching Policy Engine to Gateway...")
gateway_client.update_gateway_policy_engine(
    gateway_identifier=GATEWAY_ID,
    policy_engine_arn=policy_engine['policyEngineArn'],
    mode="LOG_ONLY"  # Start in LOG_ONLY mode, will switch to ENFORCE after adding policies
)
print("✅ Policy Engine attached in LOG_ONLY mode")
print("\n💡 The engine is empty - we'll add policies using natural language in Step 3")

---

# Step 3: Generate and Create Policy from Natural Language

Let's generate a Cedar policy from natural language and create it.

## Natural Language Input

We'll use: **"Allow refunds for amounts less than $500"**

The AI will translate this into a proper Cedar policy statement.

In [ ]:
# Generate policy from natural language
nl_input = "Allow refunds for amounts less than $500"

print("📝 Generating Cedar policy from natural language...")
print(f"   Input: \"{nl_input}\"\n")

result = policy_client.generate_policy(
    policy_engine_id=policy_engine["policyEngineId"],
    name=f"nl_policy_{int(time.time())}",
    resource={"arn": GATEWAY_ARN},
    content={"rawText": nl_input},
    fetch_assets=True
)

if result.get('status') == 'GENERATED' and result.get('generatedPolicies'):
    generated_policy = result['generatedPolicies'][0]
    cedar_statement = generated_policy.get('definition', {}).get('cedar', {}).get('statement', 'N/A')
    
    print("✅ Policy generated successfully!\n")
    print("Generated Cedar Policy:")
    print("=" * 60)
    print(cedar_statement)
    print("=" * 60)
else:
    print(f"⚠️  Generation failed with status: {result.get('status')}")

In [ ]:
# Create the generated policy
print("\n📝 Creating generated policy...")

if result.get('status') == 'GENERATED' and result.get('generatedPolicies'):
    generated_policy = result['generatedPolicies'][0]
    
    refund_policy = policy_client.create_policy(
        policy_engine_id=policy_engine["policyEngineId"],
        name="refund_policy",
        description="Allow refunds for amounts less than $500 (generated from natural language)",
        definition=generated_policy.get('definition', {})
    )
    
    print("✅ Policy deployed!")
    print(f"   Policy ID: {refund_policy['policyId']}")
    print(f"   Policy Name: {refund_policy['name']}")
    
    # Switch to ENFORCE mode
    print("\n🔒 Switching Policy Engine to ENFORCE mode...")
    gateway_client.update_gateway_policy_engine(
        gateway_identifier=GATEWAY_ID,
        policy_engine_arn=policy_engine['policyEngineArn'],
        mode="ENFORCE"
    )
    print("✅ Policy Engine now in ENFORCE mode")
    print("\n💡 Policy is now actively enforcing:")
    print("   - Refunds ≤ $500: ALLOWED ✅")
    print("   - Refunds > $500: DENIED ❌")
else:
    print("⚠️  No policy to create. Please run the generation cell above.")

---

# Step 4: Test Policy Enforcement

Now let's test the created policy with an AI agent to see it in action!

In [ ]:
# Test policy enforcement with AI agent
from mcp.client.streamable_http import streamablehttp_client
from strands import Agent
from strands.models import BedrockModel
from strands.tools.mcp.mcp_client import MCPClient

# Get access token
access_token = gateway_client.get_access_token_for_cognito(cognito_response['client_info'])

# Setup MCP client and agent
def get_tools(mcp_client):
    tools, token = [], None
    while True:
        result = mcp_client.list_tools_sync(pagination_token=token)
        tools.extend(result)
        if not result.pagination_token:
            break
        token = result.pagination_token
    return tools

model = BedrockModel(inference_profile_id="anthropic.claude-3-7-sonnet-20250219-v1:0", streaming=True)
mcp_client = MCPClient(
    lambda: streamablehttp_client(GATEWAY_URL, headers={"Authorization": f"Bearer {access_token}"})
)

In [ ]:
with mcp_client:
    agent = Agent(model=model, tools=get_tools(mcp_client))
    
    print("Testing Policy: Allow refunds ≤ $500\n")
    print("="*60)
    
    # Test 1: ALLOWED - $200 refund
    print("\n🧪 Test 1: $200 refund (should be ALLOWED ✅)")
    print("-" * 60)
    try:
        response = agent("Process a refund of $200 for order ORD-12345")
        print(response)
    except Exception as e:
        print(f"Result: {e}")
    
    # Test 2: DENIED - $2000 refund
    print("\n🧪 Test 2: $2000 refund (should be DENIED ❌)")
    print("-" * 60)
    try:
        response = agent("Process a refund of $2000 for order ORD-67890")
        print(response)
    except Exception as e:
        print(f"Result: {e}")
    
    print("\n" + "="*60)
    print("✅ Testing complete!")
    print("\n💡 The policy successfully enforced the $500 limit!")

---

# Step 5: Try Other Policies

Now let's try a different policy to see how enforcement changes!

We'll:
1. Delete the current policy
2. Generate a new policy with different rules
3. Deploy it
4. Test the new enforcement behavior

In [ ]:
# Delete the current policy
print("🗑️  Deleting current policy...")

try:
    policy_client.delete_policy(
        policy_engine_id=policy_engine['policyEngineId'],
        policy_id=refund_policy['policyId']
    )
    print("✅ Policy deleted")
    print("\n💡 The policy engine is now empty")
except Exception as e:
    print(f"⚠️  Error: {e}")

In [ ]:
# Generate a new policy with different rules
nl_input_2 = "Allow refunds between $100 and $1000"

print("📝 Generating new policy from natural language...")
print(f"   Input: \"{nl_input_2}\"\n")

result2 = policy_client.generate_policy(
    policy_engine_id=policy_engine["policyEngineId"],
    name=f"nl_policy_range_{int(time.time())}",
    resource={"arn": GATEWAY_ARN},
    content={"rawText": nl_input_2},
    fetch_assets=True
)

if result2.get('status') == 'GENERATED' and result2.get('generatedPolicies'):
    generated_policy2 = result2['generatedPolicies'][0]
    cedar_statement = generated_policy2.get('definition', {}).get('cedar', {}).get('statement', 'N/A')
    
    print("✅ Policy generated!\n")
    print("Generated Cedar Policy:")
    print("=" * 60)
    print(cedar_statement)
    print("=" * 60)
    
    # Deploy the new policy
    print("\n📝 Deploying new policy...")
    refund_policy_v2 = policy_client.create_or_get_policy(
        policy_engine_id=policy_engine["policyEngineId"],
        name="refund_range_policy",
        description="Allow refunds between $100 and $1000",
        definition=generated_policy2.get('definition', {})
    )
    
    print("✅ New policy deployed!")
    print(f"   Policy ID: {refund_policy_v2['policyId']}")
    print("\n💡 New enforcement rules:")
    print("   - Refunds < $100: DENIED ❌")
    print("   - Refunds $100-$1000: ALLOWED ✅")
    print("   - Refunds > $1000: DENIED ❌")
else:
    print(f"⚠️  Generation failed: {result2.get('status')}")

In [ ]:
# Test the new range-based policy
from mcp.client.streamable_http import streamablehttp_client
from strands import Agent
from strands.models import BedrockModel
from strands.tools.mcp.mcp_client import MCPClient

access_token = gateway_client.get_access_token_for_cognito(cognito_response['client_info'])

def get_tools(mcp_client):
    tools, token = [], None
    while True:
        result = mcp_client.list_tools_sync(pagination_token=token)
        tools.extend(result)
        if not result.pagination_token:
            break
        token = result.pagination_token
    return tools

model = BedrockModel(inference_profile_id="anthropic.claude-3-7-sonnet-20250219-v1:0", streaming=True)
mcp_client = MCPClient(
    lambda: streamablehttp_client(GATEWAY_URL, headers={"Authorization": f"Bearer {access_token}"})
)

with mcp_client:
    agent = Agent(model=model, tools=get_tools(mcp_client))
    
    print("Testing Range-Based Policy ($100-$1000)\n")
    print("="*60)
    
    print("\n🧪 Test 1: $50 (below range - should be DENIED ❌)")
    print("-" * 60)
    try:
        response = agent("Process a refund of $50 for order ORD-001")
        print(response)
    except Exception as e:
        print(f"Result: {e}")
    
    print("\n🧪 Test 2: $500 (in range - should be ALLOWED ✅)")
    print("-" * 60)
    try:
        response = agent("Process a refund of $500 for order ORD-002")
        print(response)
    except Exception as e:
        print(f"Result: {e}")
    
    print("\n🧪 Test 3: $2000 (above range - should be DENIED ❌)")
    print("-" * 60)
    try:
        response = agent("Process a refund of $2000 for order ORD-003")
        print(response)
    except Exception as e:
        print(f"Result: {e}")
    
    print("\n" + "="*60)
    print("✅ Testing complete!")
    print("\n💡 Notice how the enforcement changed:")
    print("   - First policy: allowed ≤ $500")
    print("   - New policy: allowed $100-$1000")

---

# Step 6: Cleanup

Clean up all resources to avoid ongoing charges.

In [ ]:
# Cleanup all resources
print("🧹 Starting cleanup...\n")
print("="*60)

# Clean up Policy Engine
try:
    print("\n1️⃣ Cleaning up Policy Engine...")
    policy_client.cleanup_policy_engine(policy_engine['policyEngineId'])
    print("   ✅ Policy Engine cleanup complete")
except Exception as e:
    print(f"   ⚠️  Error: {e}")

# Clean up Gateway
try:
    print("\n2️⃣ Cleaning up Gateway...")
    gateway_client.cleanup_gateway(GATEWAY_ID, cognito_response['client_info'])
    print("   ✅ Gateway cleanup complete")
except Exception as e:
    print(f"   ⚠️  Error: {e}")

print("\n" + "="*60)
print("✅ Cleanup complete!")
print("="*60)